In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import root_scalar

# Constants
g = 9.81
sigma = 0.072
rho = 1000
nu = 1e-6
f_p = 20.0
omega_p = 2 * np.pi * f_p
h_depth = 0.005

def dispersion_relation(k):
    """Dispersion relation for gravity-capillary waves"""
    return np.sqrt(g * k + (sigma / rho) * k**3) * np.tanh(k * h_depth)

def get_gamma_exact(k):
    """Returns gamma factor"""
    return nu*k**2 * (2 + 1/(np.tanh(2*k*h_depth)*np.sinh(2*k*h_depth))) + np.sqrt(k*nu*np.sqrt(9.81*h_depth)/8)*2*k/np.sinh(2*k*h_depth)

def system_dynamics(y, t_scalar, k, A):
    """Function guards the system dynamics and will be numerically solved"""
    h, v = y
    tanh_term = np.tanh(k*h_depth)
    omega0_sq = dispersion_relation(k)**2
    gamma = get_gamma_exact(k)

    # The forcing modulates gravity specifically
    # forcing = (A * k * tanh(kh)) * cos(omega_p * t)
    forcing = (A * k * tanh_term) * np.cos(omega_p * t_scalar)

    # Dynamics of system
    dhdt = v
    dvdt = -2 * gamma * v - (omega0_sq - forcing) * h
    return [dhdt, dvdt]

def instability_measure(k, A):
    """Returns real part of eigenvalues"""
    # Period and interval
    T = 2 * np.pi / omega_p
    t = [0, T]

    # Monodromy matrix construction
    res1 = odeint(system_dynamics, [1.0, 0.0], t, args=(k, A))[-1]
    res2 = odeint(system_dynamics, [0.0, 1.0], t, args=(k, A))[-1]
    M = np.column_stack([res1, res2])
    eigenvalues = np.linalg.eigvals(M)
    return np.real(max(eigenvalues, key=lambda x: abs(x)))

# Determine precision
k_values = np.linspace(50, 1200, 200)
A_max = 200
A_steps = 100 # Number of vertical slices to check for each k

sh_boundaries = []
h_boundaries = []

print("Scanning for all tongue boundaries...")

for k_val in k_values:
    # We create a function for root finding that is 0 at the boundary

    scan_A = np.linspace(0, A_max, A_steps)
    eigenvalues = []

    for a in scan_A:
        # Eigenvalues for varying values of A
        eval = instability_measure(k_val, a)
        eigenvalues.append(eval)

    # Set up loop
    val_2 = eigenvalues[0]

    for i in range(len(scan_A) - 1):
        # Boundary Condition 1 (SH): abs(eval) > 1 and eval < 0
        # Boundary Condition 2 (H): abs(eval) > 1 and eval > 0
        # Boundary Condition 3 (SH to H or reverse):
        # abs(eval) > 1 and sign(eval1) != sign(eval2)
        val_1 = val_2
        val_2 = eigenvalues[i+1]

        # Checks boundary of stability with instability
        if (abs(val_2)-1)*(abs(val_1)-1) < 0 and val_2 < 0:
            sol = root_scalar(lambda a: abs(instability_measure(k_val, a)) - 1,
                              bracket=[scan_A[i], scan_A[i+1]], method='brentq')
            sh_boundaries.append((k_val, sol.root))
        elif (abs(val_2)-1)*(abs(val_1)-1) < 0 and val_2 > 0:
            sol = root_scalar(lambda a: abs(instability_measure(k_val, a)) - 1,
                              bracket=[scan_A[i], scan_A[i+1]], method='brentq')
            h_boundaries.append((k_val, sol.root))

        # Checks boundary of subharmonic instability with harmonic instability
        # in both directions
        if abs(val_2) > 1 and val_2*val_1 < 0:
            try:
                sol = root_scalar(lambda a: abs(instability_measure(k_val, a)) - 1,
                              bracket=[scan_A[i], scan_A[i+1]], method='brentq')
                a_val = sol.root

                # Eigenvalue is checked again because points could appear in
                # the wrong group close to the boundary
                if instability_measure(k_val, a_val) < 0:
                    sh_boundaries.append((k_val, a_val))
                elif instability_measure(k_val, a_val) > 0:
                    h_boundaries.append((k_val, a_val))
            except ValueError:
                pass

# Points could still appear in the wrong group close to the boundary
# This places them in the correct group by using how many boundary points
# are below some boundary point
import matplotlib.pyplot as plt
h_list_new = h_boundaries[:]
sh_list_new = sh_boundaries[:]
for i in sh_list_new:
    count = 0
    for j in h_list_new:
        if i[0] == j[0] and i[1] > j[1]:
            count += 1
    if count % 2 == 1:
        h_list_new.append(i)
        sh_list_new.remove(i)

for j in h_list_new:
    count = 0
    for i in sh_list_new:
        if j[0] == i[0] and j[1] > i[1]:
            count += 1
    if count % 2 == 1:
        sh_list_new.append(j)
        h_list_new.remove(j)

# Plotting
k_sh_plot, A_sh_plot = zip(*sh_list_new)
k_h_plot, A_h_plot = zip(*h_list_new)
k_sh_plot, A_sh_plot = np.array(k_sh_plot), np.array(A_sh_plot)
k_h_plot, A_h_plot = np.array(k_h_plot), np.array(A_h_plot)
plt.figure(figsize=(10, 7))
plt.scatter(k_sh_plot, A_sh_plot/g, s=5, color="green", label="Subharmonic Stability Boundary")
plt.scatter(k_h_plot, A_h_plot/g, s=5, color="red", label="Harmonic Stability Boundary")
plt.legend()


# Formatting
#plt.title(f"Faraday Instability Tongues at $f_d$ = {f_d} Hz")
plt.xlabel("Wavenumber $k$ (rad/m)")
plt.ylabel("Gamma $\Gamma = a_d/g$")
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.show()